In [ ]:
#Configuration initiale & import
import importlib
import sys
import duckdb
import pandas as pd
import os

sys.path.insert(0, r"C:\Users\admin\work\idf_project\notebooks")
importlib.invalidate_caches()  

import config as cfg
files = os.listdir(r"C:\Users\admin\work\idf_project\notebooks")
print(cfg.IDF_DEPS)

In [ ]:
# Read et premier check du fichier BPE (Base permanente des équipements)
bpe = pd.read_csv(
    r"C:\Users\admin\work\idf_project\data\raw\bpe\BPE24\BPE24.csv",
    sep=";",        
    nrows=5,        
    ncolumns = 90,
    dtype=str,      
    encoding="utf-8"
)

print(bpe.shape)
print(bpe.columns.tolist())
bpe.head()

In [ ]:
# Read et premier check du fichier filosofi (Revenus fiscaux)
filo = pd.read_csv(
    r"C:\Users\admin\Documents\GIS data\idf_project\data\raw\filosofi\BASE_TD_FILO_IRIS_2021_DEC_CSV\BASE_TD_FILO_IRIS_2021_DEC.csv",
    sep=";",
    nrows=5,
    dtype=str,
    encoding="utf-8"
)

print(filo.shape)
print(filo.columns.tolist())
filo.head()

In [ ]:
# Read et premier check du fichier census (Population)
census = pd.read_csv(
    r"C:\Users\admin\Documents\GIS data\idf_project\data\raw\census\base-ic-evol-struct-pop-2022_csv\base-ic-evol-struct-pop-2022.CSV",
    sep=";",
    nrows=10,
    dtype=str,
    encoding="utf-8"
)

print(census.shape)
print(census.columns.tolist())
census.head()

print(census.columns.tolist())


In [ ]:
# Premier check avec Duckdb du fichier parquet d'accessibilité aux équipements par communes (divisées en grilles)
result = duckdb.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{ACCESS_PATH.as_posix()}')
""").df()

print(result)

sample = duckdb.sql(f"""
    SELECT * FROM read_parquet('{ACCESS_PATH.as_posix()}')
    LIMIT 5
""").df()

sample.head()

In [ ]:
type_counts = duckdb.sql(f"""
    SELECT typeeq_id, domaine, COUNT(*) as n_cells
    FROM read_parquet('{cfg.ACCESS_PATH.as_posix()}')
    GROUP BY typeeq_id, domaine
    ORDER BY domaine, typeeq_id
""").df()

print(type_counts.to_string())

In [ ]:
# Premier count des équipements en IDF
bpe_full = pd.read_csv(cfg.BPE_PATH, sep=";", dtype=str, low_memory=False)
bpe_idf = bpe_full[bpe_full['DEP'].isin(cfg.IDF_DEPS)]

print(f"Total IDF facilities: {len(bpe_idf)}")
print(f"\nFacilities by DEP:")
print(bpe_idf['DEP'].value_counts())
print(f"\nSample TYPEQU codes:")
print(bpe_idf['TYPEQU'].value_counts().head(20))

In [ ]:
target_types = list(cfg.FACILITY_TYPES.keys())
print("Checking BPE for our target types:")
print(bpe_idf[bpe_idf['TYPEQU'].isin(target_types)]['TYPEQU'].value_counts())

print("\nChecking accessibility dataset for our target types:")
access_check = duckdb.sql(f"""
    SELECT typeeq_id, COUNT(*) as n_cells
    FROM read_parquet('{cfg.ACCESS_PATH.as_posix()}')
    WHERE typeeq_id IN ('D201', 'D101', 'D301', 'C101', 'E107')
    GROUP BY typeeq_id
""").df()
print(access_check)

nomen = pd.read_excel(
    list((cfg.RAW / 'bpe').glob('*.csv'))[0]  
)
print("\nNomenclature columns:")
print(nomen.columns.tolist())
print(nomen.head(10))

In [ ]:
print("=== Health/Social facilities in IDF (Domain D) ===")
d_types = bpe_idf[bpe_idf['TYPEQU'].str.startswith('D')]['TYPEQU'].value_counts()
print(d_types.to_string())

print("\n=== Education facilities (B domain) ===")
b_types = bpe_idf[bpe_idf['TYPEQU'].str.startswith('B')]['TYPEQU'].value_counts()
print(b_types.to_string())

print("\n=== Transport (E domain) ===")
e_types = bpe_idf[bpe_idf['TYPEQU'].str.startswith('E')]['TYPEQU'].value_counts()
print(e_types.to_string())

In [ ]:
nomen = pd.read_csv(
    cfg.RAW / 'bpe' / 'BPE24_table_passage.csv',
    sep=';',
    dtype=str,
    encoding='utf-8'
)

print(nomen.columns.tolist())
print(nomen.head(20))

In [ ]:
# Join nomenclature against our intersection results
# First rerun the cross-reference and collect all matching codes

all_matching = duckdb.sql(f"""
    SELECT DISTINCT typeeq_id 
    FROM read_parquet('{cfg.ACCESS_PATH.as_posix()}')
    WHERE typeeq_id LIKE 'D%' 
       OR typeeq_id LIKE 'B%' 
       OR typeeq_id LIKE 'E%'
""").df()['typeeq_id'].tolist()

# Filter to codes that also exist in BPE IDF
bpe_codes = set(bpe_idf['TYPEQU'].unique())
matching_both = [c for c in all_matching if c in bpe_codes]

# Now decode with nomenclature
decoded = (
    nomen[nomen['TYPEQU'].isin(matching_both)]
    [['TYPEQU', 'Libelle_TYPEQU', 'Libelle_SDOM', 'Libelle_DOM']]
    .merge(
        bpe_idf['TYPEQU'].value_counts().reset_index()
        .rename(columns={'TYPEQU': 'TYPEQU', 'count': 'n_facilities_idf'}),
        on='TYPEQU'
    )
    .sort_values('n_facilities_idf', ascending=False)
)

print(decoded.to_string())

In [ ]:
# Search for schools across all domains
schools = nomen[nomen['Libelle_TYPEQU'].str.contains('ÉCOLE|ECOLE|SCOLAIRE|PRIMAIRE|MATERNELLE', 
                                                       case=False, na=False)]
print(schools[['TYPEQU','Libelle_TYPEQU','Libelle_DOM']].to_string())

# Also check C domain specifically
c_types = nomen[nomen['TYPEQU'].str.startswith('C')]
print("\nAll C domain types:")
print(c_types[['TYPEQU','Libelle_TYPEQU']].to_string())

In [ ]:
# Check which C codes exist in accessibility dataset
acc_c = duckdb.sql(f"""
    SELECT DISTINCT typeeq_id 
    FROM read_parquet('{cfg.ACCESS_PATH.as_posix()}')
    WHERE typeeq_id LIKE 'C%'
""").df()['typeeq_id'].tolist()

# Cross with BPE IDF
bpe_c = bpe_idf[bpe_idf['TYPEQU'].str.startswith('C')]['TYPEQU'].value_counts().reset_index()
bpe_c.columns = ['TYPEQU', 'n_idf']

# Decode with nomenclature
decoded_c = nomen[nomen['TYPEQU'].isin(acc_c)].merge(bpe_c, on='TYPEQU')
print("C codes in BOTH accessibility and BPE IDF:")
print(decoded_c[['TYPEQU','Libelle_TYPEQU','n_idf']].sort_values('n_idf', ascending=False).to_string())